In [113]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_absolute_error

In [114]:
df = pd.read_csv(os.path.join('..', 'backend', 'data', 'vitamin_deficiency_disease_dataset_20260123.csv'))

In [115]:

df.head()

,age,gender,bmi,smoking_status,alcohol_consumption,exercise_level,diet_type,sun_exposure,income_level,latitude_region,...,has_night_blindness,has_fatigue,has_bleeding_gums,has_bone_pain,has_muscle_weakness,has_numbness_tingling,has_memory_problems,has_pale_skin,disease_diagnosis,has_multiple_deficiencies
0,79,Male,24.8,Former,NaN,Active,Vegetarian,High,High,Mid,...,0,0,0,0,0,0,0,0,Healthy,0
1,77,Female,39.9,Former,Moderate,Light,Omnivore,Low,Low,Low,...,0,0,0,1,0,0,0,0,Rickets_Osteomalacia,0
2,24,Male,26.4,Former,Heavy,Moderate,Omnivore,Low,High,High,...,1,0,0,0,0,0,0,0,Healthy,0
3,69,Male,23.1,Never,Heavy,Moderate,Vegetarian,High,Low,Low,...,0,0,0,0,0,1,1,0,Anemia,0
4,63,Male,29.6,Never,NaN,Moderate,Vegetarian,Moderate,High,Low,...,0,0,0,0,0,0,0,0,Healthy,0


In [116]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 34 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   age                        4000 non-null   int64  
 1   gender                     4000 non-null   object 
 2   bmi                        4000 non-null   float64
 3   smoking_status             4000 non-null   object 
 4   alcohol_consumption        2722 non-null   object 
 5   exercise_level             4000 non-null   object 
 6   diet_type                  4000 non-null   object 
 7   sun_exposure               4000 non-null   object 
 8   income_level               4000 non-null   object 
 9   latitude_region            4000 non-null   object 
 10  vitamin_a_percent_rda      4000 non-null   float64
 11  vitamin_c_percent_rda      4000 non-null   float64
 12  vitamin_d_percent_rda      4000 non-null   float64
 13  vitamin_e_percent_rda      4000 non-null   float

In [117]:
df.isnull().mean()

,0
age,0.0000
gender,0.0000
bmi,0.0000
smoking_status,0.0000
alcohol_consumption,0.3195
exercise_level,0.0000
diet_type,0.0000
sun_exposure,0.0000
income_level,0.0000
latitude_region,0.0000


In [118]:
df = df[['gender',
    'smoking_status',
    'alcohol_consumption',
    'exercise_level',
    'diet_type',
    'sun_exposure',
    'income_level',
    'latitude_region',
    'symptoms_list',
    'vitamin_a_percent_rda',
    'vitamin_c_percent_rda',
    'vitamin_d_percent_rda',
    'vitamin_e_percent_rda',
    'vitamin_b12_percent_rda',
    'folate_percent_rda',
    'calcium_percent_rda',
    'iron_percent_rda']]
df = df.rename(columns={'income_level': 'stress_level'})

In [119]:
df.head()

,gender,smoking_status,alcohol_consumption,exercise_level,diet_type,sun_exposure,income_level,latitude_region,symptoms_list,vitamin_a_percent_rda,vitamin_c_percent_rda,vitamin_d_percent_rda,vitamin_e_percent_rda,vitamin_b12_percent_rda,folate_percent_rda,calcium_percent_rda,iron_percent_rda
0,Male,Former,NaN,Active,Vegetarian,High,High,Mid,NaN,119.1,147.3,152.88,97.5,102.5,188.9,108.3,97.4
1,Female,Former,Moderate,Light,Omnivore,Low,Low,Low,bone_pain,85.7,57.5,32.76,82.7,62.6,51.0,42.6,102.5
2,Male,Former,Heavy,Moderate,Omnivore,Low,High,High,dry_skin;night_blindness,48.3,152.1,94.99,169.3,136.2,116.6,136.3,86.4
3,Male,Never,Heavy,Moderate,Vegetarian,High,Low,Low,numbness_tingling;memory_problems,75.8,51.0,51.48,85.7,31.8,66.5,76.5,60.8
4,Male,Never,NaN,Moderate,Vegetarian,Moderate,High,Low,NaN,93.3,111.5,62.90,155.6,72.6,124.9,69.4,71.9


In [120]:
df['alcohol_consumption'] = df['alcohol_consumption'].fillna("Unknown")
df['symptoms_list'] = df['symptoms_list'].fillna("")

# remove string "NaN" or "nan"
df['symptoms_list'] = df['symptoms_list'].replace(['NaN', 'nan'], '')

# clean separators
df['symptoms_list'] = df['symptoms_list'].str.replace('[,;]', ' ', regex=True)

# remove extra spaces
df['symptoms_list'] = df['symptoms_list'].str.strip()

In [121]:
df['symptoms_list'] = df['symptoms_list'].str.lower()

In [122]:
vitamin_cols = [
    'vitamin_a_percent_rda',
    'vitamin_c_percent_rda',
    'vitamin_d_percent_rda',
    'vitamin_e_percent_rda',
    'vitamin_b12_percent_rda',
    'folate_percent_rda',
    'calcium_percent_rda',
    'iron_percent_rda'
]


In [123]:
X = df.drop(columns=vitamin_cols)
y = df[vitamin_cols]


In [124]:
ordinal_cols = ['exercise_level', 'sun_exposure', 'stress_level']

ordinal_categories = [
    ['Sedentary', 'Light', 'Moderate', 'Active'],
    ['Low', 'Moderate', 'High'],
    ['Low', 'Middle', 'High']
]

nominal_cols = [
    'gender',
    'smoking_status',
    'alcohol_consumption',
    'diet_type',
    'latitude_region'
]

text_col = 'symptoms_list'

In [125]:
preprocessor = ColumnTransformer(
    transformers=[
        ('ord', OrdinalEncoder(categories=ordinal_categories), ordinal_cols),
        ('nom', OneHotEncoder(handle_unknown='ignore'), nominal_cols),
        ('txt', TfidfVectorizer(), text_col)
    ]
)

In [126]:
from xgboost import XGBRegressor
pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('model',XGBRegressor(n_estimators=200,learning_rate=0.06,max_depth=4,subsample=0.8,colsample_bytree=0.8,random_state=42))
])


In [127]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [128]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('ord',
                                                  OrdinalEncoder(categories=[['Sedentary',
                                                                              'Light',
                                                                              'Moderate',
                                                                              'Active'],
                                                                             ['Low',
                                                                              'Moderate',
                                                                              'High'],
                                                                             ['Low',
                                                                              'Middle',
                                                                              'High']]),
                                                  ['exercise_level',
                                                   'sun_exposure',
                                                   'income_level']),
                                                 ('nom',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['gender', 'smoking_status',
                                                   'alcohol_consumption',
                                                   'diet_type',...
                              feature_types=None, feature_weights=None,
                              gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None, learning_rate=0.06,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=4, max_leaves=None,
                              min_child_weight=None, missing=nan,
                              monotone_constraints=None, multi_strategy=None,
                              n_estimators=200, n_jobs=None,
                              num_parallel_tree=None, ...))])

In [129]:
y_pred = pipeline.predict(X_test)

In [130]:
print("MAE:", mean_absolute_error(y_test, y_pred))

MAE: 18.840259552001953


In [131]:
from sklearn.metrics import r2_score

r2 = r2_score(y_test, y_pred)
print("R2 Score:", r2)

R2 Score: 0.5403680801391602


In [132]:
input_data = pd.DataFrame([{
    'age': 25,
    'gender': 'Male',
    'bmi': 23.5,
    'smoking_status': 'Never',
    'alcohol_consumption': 'Moderate',
    'exercise_level': 'Moderate',
    'diet_type': 'Vegetarian',
    'sun_exposure': 'Low',
    'stress_level': 'Middle',
    'latitude_region': 'Low',
    'symptoms_list': 'fatigue bone pain'
}])

In [133]:
pred = pipeline.predict(input_data)

In [134]:
vitamin_cols = [
    'vitamin_a_percent_rda',
    'vitamin_c_percent_rda',
    'vitamin_d_percent_rda',
    'vitamin_e_percent_rda',
    'vitamin_b12_percent_rda',
    'folate_percent_rda',
    'calcium_percent_rda',
    'iron_percent_rda'
]

nice_names = {
    'vitamin_a_percent_rda': 'Vitamin A Intake (%)',
    'vitamin_c_percent_rda': 'Vitamin C Intake (%)',
    'vitamin_d_percent_rda': 'Vitamin D Intake (%)',
    'vitamin_e_percent_rda': 'Vitamin E Intake (%)',
    'vitamin_b12_percent_rda': 'Vitamin B12 Intake (%)',
    'folate_percent_rda': 'Folate Intake (%)',
    'calcium_percent_rda': 'Calcium Intake (%)',
    'iron_percent_rda': 'Iron Intake (%)'
}

for i, col in enumerate(vitamin_cols):
    print(f"{nice_names[col]} → {round(pred[0][i], 2)}")

Vitamin A Intake (%) → 92.12999725341797
Vitamin C Intake (%) → 89.54000091552734
Vitamin D Intake (%) → 50.04999923706055
Vitamin E Intake (%) → 96.61000061035156
Vitamin B12 Intake (%) → 62.81999969482422
Folate Intake (%) → 59.88999938964844
Calcium Intake (%) → 94.27999877929688
Iron Intake (%) → 65.0199966430664
